In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
!ls $path

In [ ]:
# Above command to check name of csv file with terminal.
# Task 1: Write your code here:
import os, pandas as pd
df = pd.read_csv(os.path.join(path, 'Q1_data.csv'))

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
import seaborn as sns, matplotlib.pyplot as plt
sns.set_style('whitegrid') # purely for aesthetics :D

sns.histplot(df['Delivery_Time'])
plt.title("Target [Delivery_Time] Distribution")

In [ ]:
# Task 1: Write your code here:
df.drop(['Order_ID'], axis=1, inplace=True)
df.head() # displaying df after dropping to confirm if row dropped.

In [ ]:
# Task 2: Write your code here:
df.isnull().sum()

In [ ]:
# Task 2 CONTD.
# From output of above cell we can see target [delivery time] is missing in 106 rows. Cannot inpute that; (can't make assumptions.)
# Can see courier-exp (numeric) has 49 rows missing, time of day (categorical) has 50, traffic lvl (categorical) has 55, NEXT LINE
# weather (Categorical) has 55. Rest none.

df['Courier_Experience_yrs'].plot(kind='hist')
plt.title("Numeric Column Distro [Courier_exp]")
# Will impute categorical with mode and numeric with mean (since fairly balanced, not heavy outliers. as shown by output of this cell).

In [ ]:
# Task 2 CONTD. Showing value counts for each categorical column, just in case.
df['Weather'].value_counts(), df['Traffic_Level'].value_counts(), df['Time_of_Day'].value_counts(), df['Vehicle_Type'].value_counts()

In [ ]:
# Task 2 CONTD. Now imputing missing values for categorical columns.
cat_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
for col in cat_cols:
  df[col]  = df[col].fillna(df[col].value_counts().sort_values(ascending=False).index[0])
# .mode() didn't work for some reason so i'm doing it this way. value_counts -> descending order -> select the first one.
df.isnull().sum()
# outputting to see if null now or not.

In [ ]:
# Task 2 CONTD: Will first drop rows with missing target values, then impute courier_exp w mean.

df = df[df['Delivery_Time'] >= 0] #checking >= 0 cuz here shown in target distro that >= 0. better than checking if null/NaN.
df.isnull().sum(), len(df) # down from 1663 entries to 1557. fine.

In [ ]:
# Task 2 CONTD. Now impute courier exp w mean.
df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].mean(), inplace=True)
df.isnull().sum() # ignore warning. annoying.
# finally all rows cleaned up.

In [ ]:
# Task 3: Write your code here:
print("No of duplicateds:",df.duplicated().sum(), "Rows w dupes included", len(df)) # 557 duplicates shown
df = df.drop_duplicates() # 557 duplicates dropped
print("after dupes removed", len(df))
# ideally should impute after dropping dupes, but here doing it in order lab asked

In [ ]:
weather_unique = df['Weather'].unique()
original_features = df.columns
weather_unique
for w in weather_unique:
  df[f'Weather_{str(w)}'] = (df['Weather'] == w).astype('int')
df

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder
enc = LabelEncoder()
# for simplicity iterating thru all catergorical cols and labelencoding; if time then later will add onehot.
# NOTE: ADDED ONEHOT FOR WEATHER CATEGORY. AT LEAST PARTIAL BONUS, PLEASE?
# DROPPING ORIGINAL WEATHER COLUMN since now redundant. will do in subsequent cell.
for col in cat_cols:
  df[col] = enc.fit_transform(df[col]) # and fitting/transforming on same labelenc objects cuz in labs we were never shown that
  # original labels are used later, and such stuff has been done in labs

In [ ]:
df = df.drop(['Weather'], axis=1)

In [ ]:
# just visualising now if all encoded properly
df[:100] # YOU CAN SEE WEATHER IS ONEHOT ENCODED. BONUS please.

In [ ]:
# Task 5: Write your code here: Now standarscaler
from sklearn.preprocessing import StandardScaler
# i understand that usually scaling is done AFTER traintestsplit to prevent data leakage,
# but here doing it in the order lab instructs me to.
scale = StandardScaler()
features = df.drop(['Delivery_Time'], axis=1).columns
df[features] = scale.fit_transform(df[features])
df.head()

In [ ]:
# # Task 6: Write your code here: had already checked target distro but will check again (bonus?)
df['Delivery_Time'].plot(kind='hist') # now somewhat skewed right but not too bad.

In [ ]:
# Task 1: Write your code here:
X = df.drop(['Delivery_Time'], axis=1) # not inplace=True so will just return copy of df without target col
y = df['Delivery_Time']
X.head()

In [ ]:
# Task 2,3,4,5: Write your code here:
# Since here regression, KFold. StratifiedKFold for class regression when class imbalance.
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

import numpy as np

maes = []

rf = RandomForestRegressor()
kf = KFold(n_splits = 5, shuffle=True, random_state=42) #randomstate for reprod'bility


for tr_idx, val_idx in kf.split(X, y):
    X_tr = X.iloc[tr_idx]
    y_tr = y.iloc[tr_idx]

    X_val = X.iloc[val_idx]
    y_val = y.iloc[val_idx]

    rf.fit(X_tr, y_tr)
    y_pred = rf.predict(X_val)
    mae = mean_absolute_error(y_val, y_pred)
    maes.append(mae)
print(f"Avg MAE across all folds: {np.mean(maes):.3f}")
print("REMEMBER THIS. COMPARED IN BONUS PART.")

In [ ]:
# Task 1: Write your code here:
feature_imps = rf.feature_importances_ # gives feature importance in order of labels
feature_names = X.columns # labels, in order

f_imps = pd.Series(feature_imps, index=feature_names).sort_values(ascending=False)
f_imps.plot(kind='barh')
plt.gca().invert_yaxis() # cuz otherwise most important is at bottom
plt.title("Feature Importances, RF Regressor Model")

In [ ]:
# Task 2: Write your code here:
sns.histplot(y_pred) # without
plt.title("Delivery Time Prediction Distributions")
# similar to original: normal-distro like, but slightly skewed to right

In [ ]:
# Task Bonus: Write your code here:
#!pip install catboost
# comment/uncomment above line. now commented since reran cell; reinstalling wastes time. it gets stuck saying req alr satisfied
from catboost import CatBoostRegressor

maes = []

rf = RandomForestRegressor()
cb = CatBoostRegressor(verbose=0)

kf = KFold(n_splits = 5, shuffle=True, random_state=42)
for tr_idx, val_idx in kf.split(X, y):
    X_tr = X.iloc[tr_idx]
    y_tr = y.iloc[tr_idx]

    X_val = X.iloc[val_idx]
    y_val = y.iloc[val_idx]

    rf.fit(X_tr, y_tr)
    cb.fit(X_tr, y_tr)
    y_pred_rf = rf.predict(X_val)
    y_pred_cb = cb.predict(X_val)

    y_pred = (y_pred_rf+y_pred_cb)/2
    #print(type(y_pred), y_pred.shape)
    mae = mean_absolute_error(y_val, y_pred)
    maes.append(mae)
print(f"Avg MAE across all folds (AVG OF RF AND CATBOOST): {np.mean(maes):.3f}")

In [ ]:
# done.